Full disclaimer,,,
 
I ripped tf outta this MIT lecture [here](https://ocw.mit.edu/courses/15-071-the-analytics-edge-spring-2017/pages/linear-regression/moneyball-the-power-of-sports-analytics/) where they did it in R.  

But I made it cooler and better. 

# ISBE 
* I - Inspect
* S - Select
* B - Build
* E - Evaluate 

---
# ⚾ The Story: Moneyball (2002 Oakland A's)

**The problem.** It's the end of the 2001 season. The Oakland Athletics just lost their three best players (Jason Giambi, Johnny Damon, and closer Jason Isringhausen) to teams that could pay them more. The A's have one of the **smallest payrolls in baseball, around \$40M**. The New York Yankees spend roughly **three times that**.

Rich teams can buy the best players. Poor teams can't. So in theory the rich teams should win every year.

**The idea.** A's General Manager **Billy Beane** and his analyst **Paul DePodesta** chose a different approach. Instead of trusting scouts' gut feelings ("he *looks* like a ballplayer"), they used **data** to ask:

> *What actually wins baseball games, and which of those skills are cheap to buy because everyone else undervalues them?*

This approach is called **sabermetrics**, a term popularised by the baseball writer Bill James.

**The plan we'll follow in this notebook.** DePodesta reduced the whole season to a chain of questions. Each one can be answered with a regression model:

```
 Make the playoffs  ←  Win ~95 games  ←  Big enough run differential (RS − RA)
                                               ↑                    ↑
                                       Runs Scored (RS)      Runs Allowed (RA)
                                         ↑ OBP, SLG?          ↑ OOBP, OSLG?
```

1. **How many wins do we need** to make the playoffs?
2. **How big a run differential** (runs scored − runs allowed) do we need to get those wins?
3. **Which hitting stats** drive *runs scored*? (Is it batting average, like the scouts believe?)
4. **Which pitching/defence stats** drive *runs allowed*?
5. Use those models to **predict the 2002 season** and to **find undervalued players**.

**Our time machine.** To be fair to Billy Beane, we can only use information he had *before* the 2002 season. That's why the data pipeline below **throws away every season from 2002 onward**.

We'll use the **ISBE** workflow at every step: **Inspect** the data, **Select** variables, **Build** a model, **Evaluate** it.

## 🧰 Step 0: Load our tools
Run this cell to import everything we need:
* `pandas` handles data tables.
* `plotly` draws interactive charts.
* `statsmodels` is our main regression tool. It gives us coefficients **and** p-values, which is what an analyst like DePodesta wants.
* `sklearn` is used for train/test splitting and scoring.

In [ ]:
# PANDAS IS FOR DATA WRANGLING
import pandas as pd

# PLOTLY MAKES INTERACTIVE CHARTS
import plotly.express as px

# METRICS HELP US SCORE OUR MODEL
from sklearn import metrics

# HELP US SPLIT OUR DATA INTO TESTING AND TRAINING
from sklearn.model_selection import train_test_split

# Good ol statsmodels
import statsmodels.api as sm

# Specific root mean squared error for stats models
from statsmodels.tools.eval_measures import rmse

## 🔎 I: Inspect. Load the data and travel back to 2001
Every row is **one team in one season** (for example, the 2001 Oakland A's).

Look at what the code below does:
1. Reads `baseball.csv`.
2. Renames the columns: abbreviations are spelled out using a dictionary, and every label is made lower case with underscores for spaces (`RS` becomes `runs_scored`).
3. Keeps only seasons **before 2002**. This is our "time machine": we pretend we're DePodesta in the 2001–02 off-season.
4. Creates **run differential**: `runs_scored − runs_allowed`. It's stored as `runs_diff`.

> 💡 **Why run differential?** You win a baseball game by scoring more runs than the other team. Over a full 162-game season, a team that scores many more runs than it gives up should win a lot of games. Keep that idea in mind; we'll test it later.

In [ ]:
df = pd.read_csv('data/baseball.csv')

In [ ]:
# Spell out the abbreviated column names so the data is easier to read
column_names = {
    'RS': 'runs_scored',
    'RA': 'runs_allowed',
    'W': 'wins',
    'OBP': 'on_base_percentage',
    'SLG': 'slugging_percentage',
    'BA': 'batting_average',
    'G': 'games',
    'OOBP': 'opponent_obp',
    'OSLG': 'opponent_slg',
    'RankSeason': 'rank_season',
    'RankPlayoffs': 'rank_playoffs',
}
df = df.rename(columns=column_names)

# Make every column label lower case (e.g. Team -> team, Year -> year)
df.columns = df.columns.str.lower()

In [ ]:
# Time machine: keep only seasons before Moneyball (2002)
df = df[df['year'] < 2002].reset_index(drop=True)

In [ ]:
# Run differential: runs scored minus runs allowed
df['runs_diff'] = df['runs_scored'] - df['runs_allowed']

df.head()

#### Data Dictionary

| **Column**              | **Original** | **Description**                                          |
| ----------------------- | ------------ | -------------------------------------------------------- |
| **team**                | Team         | Name of the team                                         |
| **league**              | League       | Name of the league                                       |
| **year**                | Year         | Year of the season                                       |
| **runs_scored**         | RS           | Runs scored by the team                                  |
| **runs_allowed**        | RA           | Runs allowed by the team                                 |
| **wins**                | W            | Number of wins                                           |
| **on_base_percentage**  | OBP          | On-Base Percentage                                       |
| **slugging_percentage** | SLG          | Slugging Percentage                                      |
| **batting_average**     | BA           | Batting Average                                          |
| **playoffs**            | Playoffs     | Whether the team made the playoffs (`1 = Yes`, `0 = No`) |
| **rank_season**         | RankSeason   | Regular-season rank among playoff teams (blank if missed playoffs) |
| **rank_playoffs**       | RankPlayoffs | Playoff finish, 1 = won the World Series (blank if missed playoffs) |
| **games**               | G            | Number of games played                                   |
| **opponent_obp**        | OOBP         | Opponent On-Base Percentage                              |
| **opponent_slg**        | OSLG         | Opponent Slugging Percentage                             |
| **runs_diff**           | (new)        | Runs scored minus runs allowed (`runs_scored - runs_allowed`) |


> 💡 **Baseball stats in plain English** (you don't need to be a fan!)
> * **BA (Batting Average)**: hits ÷ at-bats. *How often does a batter get a hit?* This is the stat **traditional scouts loved**.
> * **OBP (On-Base Percentage)**: how often a batter **reaches base by any means** (hit, walk, hit-by-pitch). A walk counts here but not in BA.
> * **SLG (Slugging Percentage)**: total bases ÷ at-bats. Measures **power**: a home run is worth 4, a single is worth 1.
> * **OOBP / OSLG**: the same stats, but for the **opponents' hitters** against our pitchers. Lower is better for us.
>
> 🎯 **The Moneyball bet:** the market paid big money for high **batting average**, but walks (which boost OBP) were nearly free. If OBP matters more than BA for scoring runs, the A's can buy runs cheaply.

### Sanity-check the data
Before modelling, always check three things: **shape**, **nulls**, and **duplicates**.

`df.info()` covers shape and nulls in one command: the row count, plus the **non-null count** for each column (a column with fewer non-null values than rows has missing data). Duplicates get their own check.

👀 **Look for:** which columns have lots of missing values? `rank_season` and `rank_playoffs` are only filled in for playoff teams, so the blanks make sense. **`opponent_obp` and `opponent_slg`** are missing for most of the older seasons. Remember this; we'll need to drop those rows when we model runs allowed.

In [ ]:
df.info()

In [ ]:
df.duplicated().sum()

---
# 🏏 Part 1: How do we SCORE runs?
The A's can't afford the "best" hitters, so they need to know **exactly which hitting skills produce runs**. Then they can shop for those skills at a discount.

Our **dependent variable** (what we're predicting) is `runs_scored`, runs scored in a season.

# SCORING RUNS
_Lets build a Linear Regression model to see what is most important to predict how many runs we score from the data we have_

In [ ]:
df.head()

### S: Select (first attempt; spot the mistake!)
For our first try, we'll throw in **almost every numeric column** we have and see what happens. As you run the next few cells, think like a detective: *does it make sense to use these columns to predict runs scored?*

In [ ]:
dependent_variable = 'runs_scored'
independent_variables = ['year', 'on_base_percentage', 'slugging_percentage', 'batting_average',
                         'runs_allowed', 'wins', 'playoffs', 'games', 'runs_diff']

X = df[independent_variables]
y = df[dependent_variable]

X.head()

### B: Build
`statsmodels` needs us to add a **constant** (a column of 1s) so the model can learn an intercept. Then `.fit()` finds the best-fitting coefficients.

In [ ]:
import statsmodels.api as sm

## we have to add a constant to our X data (a column of just all 1's)
X = sm.add_constant(X)

# build our model
model_wins = sm.OLS(y, X).fit()

# make new predictions
y_pred = model_wins.predict(X)

### E: Evaluate
The fitted model stores everything we need to judge it. Let's pull out each piece one at a time.

**1. Coefficients** (`.params`): how much the prediction changes when that variable goes up by 1, holding the others constant. `const` is the intercept.

In [ ]:
model_wins.params

**2. p-values** (`.pvalues`): a small p-value (below 0.05) means the variable has a real effect on the prediction, not just noise.

In [ ]:
model_wins.pvalues

**3. R²** (`.rsquared`): how much of the variation in runs scored the model explains. Closer to 1 is better.

In [ ]:
model_wins.rsquared

**4. RMSE** (root mean squared error): the typical size of a prediction miss, in runs. The model doesn't store it, so we calculate it from the actual values `y` and the predictions `y_pred`.

In [ ]:
rmse(y, y_pred)

Is that miss big or small? Compare it with the average number of runs scored.

In [ ]:
y.mean()

### Put it all together: a model report
We'll check these same numbers for every model we build, so let's wrap them in a function. `print_model_report` prints the coefficients and p-values side by side, then R², RMSE, and the average of the dependent variable.

By default it scores the model on the data it was trained on. Pass in different `X` and `y` (like a test set) to score it on that data instead.

In [ ]:
def print_model_report(model, X=None, y=None):
    # Coefficient and p-value for each variable
    report = pd.DataFrame({'coefficient': model.params, 'p_value': model.pvalues})
    print(report.round(4))
    print()

    # By default, score the model on the data it was trained on
    # (model.model.exog holds the X values; model.model.endog holds the actual y values)
    if X is None:
        X = model.model.exog
        y = model.model.endog

    # How well the model fits
    y_pred = model.predict(X)
    print('R-squared:', round(metrics.r2_score(y, y_pred), 4))
    print('RMSE:', round(rmse(y, y_pred), 2))
    print(f'Average {model.model.endog_names}:', round(y.mean(), 2))

In [ ]:
print_model_report(model_wins)

# What is wonky about the Model Report above....?
0. What are we trying to predict...?
0. What are we using to predict that? 
0. What are the assumptions of Linear Regression. 

<details>
<summary><b>🧑‍🏫 Click for the answer (try first!)</b></summary>

* **We're cheating (data leakage).** `runs_diff` is literally `runs_scored − runs_allowed`. If you know `runs_diff` and `runs_allowed`, you can calculate `runs_scored` exactly, with no baseball knowledge needed. The model "predicts" perfectly because we handed it the answer.
* **Wrong direction of cause.** `wins` and `playoffs` are *results* of scoring runs, not causes. Billy Beane can't go out and buy "wins" to make his team score more runs.
* **Not useful for decisions.** `year` and `games` (games played) aren't skills you can buy from a player.
* **Assumption check:** linear regression assumes the predictors aren't near-copies of each other (no **multicollinearity**). `runs_scored`, `runs_allowed`, `runs_diff`, and `wins` are tangled together.

**Rule of thumb:** only use predictors that (1) you would **know before the outcome happens**, and (2) you can actually **act on**. For a GM, that means player skills like OBP, SLG, and BA.
</details>

### 📊 Visualise each variable against runs scored
A picture often shows the problem faster than a summary table. Each mini-plot shows one variable against `runs_scored`, with a red best-fit line.

👀 **Look for:** which plots show a sensible, real-world relationship between a **player skill** and runs (`on_base_percentage`, `slugging_percentage`, `batting_average`)?

Notice that `runs_diff` and `runs_allowed` each look only loosely related to `runs_scored` on their own. The leakage comes from using them **together**: `runs_scored = runs_diff + runs_allowed` exactly. No single scatter plot can show that, which is why the regression looked perfect even though none of these plots do.

In [ ]:
# Reshape to long format: one row per (variable, value) pair, so each variable gets its own panel
plot_df = df.melt(id_vars='runs_scored', value_vars=['year', 'on_base_percentage', 'slugging_percentage'])

fig = px.scatter(plot_df, x='value', y='runs_scored', facet_col='variable',
                 trendline='ols', trendline_color_override='red', opacity=0.4, height=350)
fig.update_traces(marker_size=4)
fig.update_xaxes(matches=None, title_text='')  # each variable gets its own x-axis scale
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[1]))  # 'variable=year' -> 'year'
fig.show()

In [ ]:
# Reshape to long format: one row per (variable, value) pair, so each variable gets its own panel
plot_df = df.melt(id_vars='runs_scored', value_vars=['batting_average', 'runs_allowed', 'wins'])

fig = px.scatter(plot_df, x='value', y='runs_scored', facet_col='variable',
                 trendline='ols', trendline_color_override='red', opacity=0.4, height=350)
fig.update_traces(marker_size=4)
fig.update_xaxes(matches=None, title_text='')  # each variable gets its own x-axis scale
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[1]))  # 'variable=year' -> 'year'
fig.show()

In [ ]:
# Reshape to long format: one row per (variable, value) pair, so each variable gets its own panel
plot_df = df.melt(id_vars='runs_scored', value_vars=['playoffs', 'games', 'runs_diff'])

fig = px.scatter(plot_df, x='value', y='runs_scored', facet_col='variable',
                 trendline='ols', trendline_color_override='red', opacity=0.4, height=350)
fig.update_traces(marker_size=4)
fig.update_xaxes(matches=None, title_text='')  # each variable gets its own x-axis scale
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[1]))  # 'variable=year' -> 'year'
fig.show()

#### Okay lets fix it with proper signals aka variables 

We keep only the three **hitting skills** a GM could actually shop for: **OBP, SLG, and BA**. This is the exact question the A's front office faced: *which of these three should we pay for?*

In [ ]:
dependent_variable = 'runs_scored'
independent_variables = ['on_base_percentage', 'slugging_percentage', 'batting_average']

X = df[independent_variables]
y = df[dependent_variable]

## we have to add a constant to our X data (a column of just all 1's)
X = sm.add_constant(X)

# build our model
model_wins = sm.OLS(y, X).fit()

print_model_report(model_wins)

# Okay, with the model report above for OBP, SLG, and BA. 
1. Our r^2 score looks great (0.93) 
2. However, look closely at the coefficients. 
3. Batting Average (which is a good thing) is negative,,, meaning the higher the batting average is the less runs you score. 
4. That does not make sense at all.... 


## Why is this happening.  

In [ ]:
df[independent_variables].corr().round(4)

👀 **Look at the correlation table.** All three stats are strongly correlated with each other (0.8 or higher). That makes sense: a player who gets lots of hits usually also gets on base a lot and hits for power.

This is called **multicollinearity**. When predictors overlap this much, the model can't tell which one deserves the credit. It splits the credit oddly, and a coefficient can even **flip sign**. That's why BA came out *negative*. It doesn't mean hits hurt you; it means BA has almost nothing extra to add once OBP and SLG are already in the model.

<div>
<img src=https://i.imgflip.com/80a3lx.jpg width="500">
</div>

# We are going to do a lot of regression models. So lets put it into a resuable function.

### Two helper functions
* **`eval_regression`** takes the prepared `X` and `y`, splits them into **80% training / 20% testing**, fits on the training part, and prints the model report scored on the testing part. This checks whether the model works on seasons it has **never seen**, which is the real test for predicting 2002.
* **`do_moneyball`** fits the model on **all** the data, prints the model report (coefficients, p-values, R² and RMSE), then calls `eval_regression` for the held-out score. It **returns the fitted model** so we can use it to make predictions later.

In [ ]:
def eval_regression(X, y):
    # Split the data into training and test sets (80% train, 20% test)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Build the model on the training data, then score it on the testing data it has never seen
    model = sm.OLS(y_train, X_train).fit()
    print_model_report(model, X_test, y_test)

In [ ]:
def do_moneyball(input_df, independent_variables, dependent_variable):
    # Define the independent and dependent variables, plus a constant for the intercept
    X = sm.add_constant(input_df[independent_variables])
    y = input_df[dependent_variable]

    print('Predicting:', dependent_variable)
    print('Using:', independent_variables)

    # Build the model on ALL the data
    model = sm.OLS(y, X).fit()
    print('\nEVAL ON ALL DATA')
    print_model_report(model)

    # Score a model built on held-out data
    print('\nEVAL ON TESTING DATA')
    eval_regression(X, y)

    return model

In [ ]:
selected_features = ['on_base_percentage', 'slugging_percentage', 'batting_average']
target_feature = 'runs_scored'

model = do_moneyball(df, independent_variables=selected_features, dependent_variable=target_feature)

# To mitigate the multicolinearity, lets try all the combinations of the variables. 

We'll try each **pair** of stats and compare. For each run, write down:
1. **R²**: how much of the variation in runs is explained (higher is better)
2. **RMSE**: the typical miss, in runs per season (lower is better)
3. Do the **coefficients make sense** (positive) and are the **p-values small**?

| Predictors | R² | RMSE | Coefficients make sense? |
|---|---|---|---|
| BA + SLG | | | |
| BA + OBP | | | |
| OBP + SLG | | | |

In [ ]:
X_vars = ['batting_average', 'slugging_percentage']
y_var = 'runs_scored'
model_BA_SLG = do_moneyball(df, X_vars, y_var)

In [ ]:
X_vars = ['batting_average', 'on_base_percentage']
y_var = 'runs_scored'
model_BA_OBP = do_moneyball(df, X_vars, y_var)

In [ ]:
X_vars = ['on_base_percentage', 'slugging_percentage']
y_var = 'runs_scored'
model_OBP_SLG = do_moneyball(df, X_vars, y_var)


<details>
<summary><b>🧑‍🏫 Click to compare with the instructor's results</b></summary>

| Predictors | R² (all data) | RMSE | Coefficients make sense? |
|---|---|---|---|
| BA + SLG | 0.876 | ~33 runs | ✅ |
| BA + OBP | 0.832 | ~38 runs | ✅ |
| **OBP + SLG** | **0.930** | **~25 runs** | ✅ |
| OBP + SLG + BA | 0.930 | ~25 runs | ❌ BA is negative |

**OBP + SLG** matches the three-variable model's accuracy *without* the nonsense negative coefficient. Adding BA bought us nothing. **Batting average, the scouts' favourite stat, is the one we can drop.** That's the heart of Moneyball.
</details>

In [ ]:
X_vars = ['on_base_percentage', 'slugging_percentage']
y_var = 'runs_scored'
model_OBP_SLG = do_moneyball(df, X_vars, y_var)


# Using SLG and OBP have the best r^2 score with significatn p-values 
## Thus we can conclude that OBP and SLG are the best signals for scoring runs. 

Holding slugging constant, a team that improves its OBP by 0.010 (for example, from .320 to .330) is expected to score about 27 more runs in a season.

```2737.77 × 0.01 = 27.38``` more runs per season.

### 💰 Why this matters for the A's
Look at the two coefficients: **OBP ≈ 2738** and **SLG ≈ 1585**. OBP and SLG are measured on roughly similar scales, so **a point of OBP is worth noticeably more runs than a point of SLG**.

The market, though, priced players by **batting average and power**. Players who drew lots of walks (high OBP, ordinary BA) were **cheap**. DePodesta's insight: *buy on-base percentage, because nobody else is paying for it.*

### 🧪 Try the model on a real team: the 2000 Oakland A's
Let's pull the A's row from the 2000 season and plug their **OBP and SLG** into our model. Then compare the prediction with how many runs they actually scored (`runs_scored`).

In [ ]:
#so, using our model.  we can now see how many runs we should score with any on_base_percentage and slugging_percentage...
# year 2000's stats, we can see how many runs we will earn
team_filter = df.team == 'OAK'
year_filter = df.year == 2000

tmp_df = df[(team_filter) & (year_filter)].copy()
# tmp_df = tmp_df[X_vars]
tmp_df

In [ ]:
model_OBP_SLG.predict([1, 0.36, 0.458])

👀 The model predicts about **907 runs** and the 2000 A's actually scored **947**. That's within about 40 runs, using only two numbers. Not bad!

> 📝 The `[1, 0.36, 0.458]` list is `[constant, on_base_percentage, slugging_percentage]`, in the same order as the columns we trained on. The leading `1` is for the intercept.

---
# 💵 Part 2: Finding undervalued players
We now know **OBP** is a big driver of runs. The next question is the money one: *which players have a high OBP but a low salary?* Those are the players the A's can afford.

This section switches to a **player-level** dataset: one row per player per season, with salaries.

# Show me the money!

In [ ]:
# Load the player-level dataset: one row per player per season, with salary
enhanced_df = pd.read_csv('data/enhanced_moneyball_analysis_1990_2010.csv')
enhanced_df.head()

### Investigate the player data
`df.info()` again shows the row count, the **non-null count** for each column, and each column's data type.

👀 **Look for:** which columns have missing values? Which stats look familiar from the team data, and which are new (`AB`, `H`, `BB`, `HR`, `OPS`)?

In [ ]:
enhanced_df.info()

### Rename the columns to match the team data
We use the same approach as before: a dictionary spells out the abbreviations, then every label is made lower case. Columns that appear in both datasets get the **same name**, so `OBP` becomes `on_base_percentage` here too.

In [ ]:
# Spell out the abbreviated column names, using the same names as the team data
player_column_names = {
    'playerID': 'player_id',
    'playerName': 'player_name',
    'yearID': 'year',
    'teamID': 'team',
    'G': 'games',
    'AB': 'at_bats',
    'H': 'hits',
    'BB': 'walks',
    'HR': 'home_runs',
    'BA': 'batting_average',
    'OBP': 'on_base_percentage',
    'SLG': 'slugging_percentage',
    'OPS': 'on_base_plus_slugging',
}
enhanced_df = enhanced_df.rename(columns=player_column_names)

# Make every column label lower case (e.g. Salary_per_OBP -> salary_per_obp)
enhanced_df.columns = enhanced_df.columns.str.lower()

In [ ]:
# Time machine: keep only seasons before Moneyball (2002) and players with a recorded salary
pre_moneyball_df = enhanced_df[(enhanced_df['year'] < 2002) & (enhanced_df['salary'] > 0)].copy()

pre_moneyball_df.head()

Again we use our **time machine**: keep only seasons up to 2001 and players with a recorded salary. This is the "free-agent shopping list" DePodesta could have built.

### The quadrant chart
We split players into four groups using two cut-offs:
* **High OBP**: top 25% of on-base percentage
* **Low salary**: bottom 25% of pay

|  | Low salary | High salary |
|---|---|---|
| **High OBP** | 🟢 **Undervalued: buy these!** | ⚪ Good, but you pay for it |
| **Low OBP** | 🔵 Cheap and you get what you pay for | 🔴 **Overvalued: avoid** |

👀 **Look for:** how *few* players land in the green box compared with the red box. Hover over the dots to see who they are. Scroll the printed list for a familiar name: **Jeremy Giambi (2001 OAK)**, one of the real players in the Moneyball story.

In [ ]:
# Remove rows with missing on_base_percentage or salary data for the analysis
analysis_df = pre_moneyball_df.dropna(subset=['on_base_percentage', 'salary']).copy()

# Calculate percentiles to identify undervalued players
# High OBP = above 75th percentile
# Low Salary = below 25th percentile
obp_75th = analysis_df['on_base_percentage'].quantile(0.75)
salary_25th = analysis_df['salary'].quantile(0.25)

print(f"=== QUADRANT ANALYSIS CRITERIA ===")
print(f"High OBP threshold (75th percentile): {obp_75th:.3f}")
print(f"Low Salary threshold (25th percentile): ${salary_25th:,.0f}")

# Create quadrant categories for each player
def assign_quadrant(row):
    if row['on_base_percentage'] >= obp_75th and row['salary'] <= salary_25th:
        return 'High OBP, Low Salary (Green)'  # Undervalued
    elif row['on_base_percentage'] < obp_75th and row['salary'] <= salary_25th:
        return 'Low OBP, Low Salary (Blue)'
    elif row['on_base_percentage'] >= obp_75th and row['salary'] > salary_25th:
        return 'High OBP, High Salary (Gray)'
    else:
        return 'Low OBP, High Salary (Red)'  # Overvalued

analysis_df['quadrant'] = analysis_df.apply(assign_quadrant, axis=1)

# Count players in each quadrant
quadrant_counts = analysis_df['quadrant'].value_counts()
print(f"\n=== QUADRANT DISTRIBUTION ===")
for quadrant, count in quadrant_counts.items():
    print(f"{quadrant}: {count} players")

# Identify undervalued players (green quadrant)
undervalued_players = analysis_df[analysis_df['quadrant'] == 'High OBP, Low Salary (Green)'].copy()
print(f"\nNumber of undervalued players identified: {len(undervalued_players)}")

# Create interactive scatter plot with quadrant-based coloring
fig = px.scatter(
    analysis_df,
    x='salary',
    y='on_base_percentage',
    color='quadrant',  # Color by quadrant
    size='games',  # Games played as size
    size_max=8,  # Smaller maximum bubble size
    hover_data=['player_id', 'team', 'year', 'batting_average', 'slugging_percentage', 'on_base_plus_slugging'],
    title='Quadrant Analysis: OBP vs Salary (Pre-Moneyball Era)',
    labels={
        'salary': 'Salary ($)',
        'on_base_percentage': 'On-Base Percentage (OBP)',
        'games': 'Games Played',
        'quadrant': 'Quadrant'
    },
    color_discrete_map={
        'High OBP, Low Salary (Green)': 'green',
        'Low OBP, Low Salary (Blue)': 'blue',
        'High OBP, High Salary (Gray)': 'gray',
        'Low OBP, High Salary (Red)': 'red'
    }
)

# Add threshold lines
fig.add_hline(y=obp_75th, line_dash="dash", line_color="red",
              annotation_text=f"High OBP Threshold ({obp_75th:.3f})")
fig.add_vline(x=salary_25th, line_dash="dash", line_color="red",
              annotation_text=f"Low Salary Threshold (${salary_25th:,.0f})")

# Quadrant-based coloring is now handled by the main scatter plot

# Update layout
fig.update_layout(
    width=1200,
    height=600,
    showlegend=False,
    xaxis_title="Salary ($)",
    yaxis_title="On-Base Percentage (OBP)",
    title_x=0.5
)

# Add quadrant labels
fig.add_annotation(
    x=0.95, y=0.95,
    xref="paper", yref="paper",
    text="<b>High OBP<br>High Salary</b>",
    showarrow=False,
    font=dict(size=12, color="gray"),
    bgcolor="rgba(255,255,255,0.8)",
    bordercolor="gray",
    borderwidth=1
)

fig.add_annotation(
    x=0.05, y=0.95,
    xref="paper", yref="paper",
    text="<b>High OBP<br>Low Salary</b><br><span style='color:green'>UNDERVALUED</span>",
    showarrow=False,
    font=dict(size=12, color="green"),
    bgcolor="rgba(255,255,255,0.8)",
    bordercolor="green",
    borderwidth=2
)

fig.add_annotation(
    x=0.95, y=0.05,
    xref="paper", yref="paper",
    text="<b>Low OBP<br>High Salary</b><br><span style='color:red'>OVERVALUED</span>",
    showarrow=False,
    font=dict(size=12, color="red"),
    bgcolor="rgba(255,255,255,0.8)",
    bordercolor="red",
    borderwidth=1
)

fig.add_annotation(
    x=0.05, y=0.05,
    xref="paper", yref="paper",
    text="<b>Low OBP<br>Low Salary</b>",
    showarrow=False,
    font=dict(size=12, color="blue"),
    bgcolor="rgba(255,255,255,0.8)",
    bordercolor="blue",
    borderwidth=1
)

fig.show()

# Display top undervalued players if any exist
if len(undervalued_players) > 0:
    print(f"\n=== UNDERVALUED PLAYERS (High OBP (descending), Low Salary) ===\n")
    top_undervalued = undervalued_players.sort_values(['on_base_percentage'], ascending=[False])[['player_id', 'year', 'team', 'on_base_percentage', 'batting_average', 'slugging_percentage', 'on_base_plus_slugging', 'salary']]
    print(top_undervalued.to_string(index=False))
else:
    print("\nNo players meet the strict criteria for undervalued players.")
    print("Let's look at players with above-average OBP and below-average salary...")

    # Let us relax the criteria to find more undervalued players.
    # Relaxed criteria:
    # 1. Above median OBP
    # 2. Below median salary
    obp_median = analysis_df['on_base_percentage'].median()
    salary_median = analysis_df['salary'].median()

    relaxed_undervalued = analysis_df[
        (analysis_df['on_base_percentage'] >= obp_median) &
        (analysis_df['salary'] <= salary_median)
    ].copy()

    print(f"\nRelaxed criteria (above median OBP: {obp_median:.3f}, below median salary: ${salary_median:,.0f})")
    print(f"Number of players: {len(relaxed_undervalued)}")

    if len(relaxed_undervalued) > 0:
        top_relaxed = relaxed_undervalued.sort_values(['on_base_percentage'], ascending=[False])[['player_id', 'year', 'team', 'on_base_percentage', 'batting_average', 'slugging_percentage', 'on_base_plus_slugging', 'salary']]
        print(top_relaxed.to_string(index=False))

### Oakland A's vs New York Yankees in 2002
**2002** is the first season the A's built their roster with the Moneyball strategy. This chart steps past our time machine to see the result: every player from the 2002 season, with the A's in their team gold, the Yankees in their team blue, and everyone else in light gray. The red dashed lines are the same cut-offs as above (from all pre-2002 players).

👀 **Look for:** the Yankees spend big for on-base percentage. Hover over the A's dot near the high-OBP line at about $900K: that's **Scott Hatteberg**, a Moneyball signing who got on base almost as often as players paid many times more.

In [ ]:
# Keep every player from the 2002 season with a recorded salary
# (enhanced_df still has every year; pre_moneyball_df stops at 2001)
year_filter = enhanced_df['year'] == 2002
salary_filter = enhanced_df['salary'] > 0
players_2002_df = enhanced_df[year_filter & salary_filter].copy()

# Label the A's and Yankees by team; everyone else is 'Other teams'
players_2002_df['team_group'] = players_2002_df['team'].where(players_2002_df['team'].isin(['OAK', 'NYY']), 'Other teams')

fig = px.scatter(
    players_2002_df,
    x='salary',
    y='on_base_percentage',
    color='team_group',
    color_discrete_map={'Other teams': 'lightgray', 'OAK': '#EFB21E', 'NYY': '#003087'},  # A's gold, Yankees blue
    category_orders={'team_group': ['Other teams', 'OAK', 'NYY']},  # draw gray first so team colors sit on top
    hover_data=['player_name', 'team', 'batting_average', 'slugging_percentage'],
    title="Oakland A's vs New York Yankees: OBP vs Salary (2002 Season)",
    labels={
        'salary': 'Salary ($)',
        'on_base_percentage': 'On-Base Percentage (OBP)',
        'team_group': 'Team'
    },
    width=1200,
    height=600
)
fig.update_traces(marker_size=10)

# Same cut-offs as the full quadrant chart (from pre-2002 players)
fig.add_hline(y=obp_75th, line_dash="dash", line_color="red")
fig.add_vline(x=salary_25th, line_dash="dash", line_color="red")

fig.update_layout(title_x=0.5)
fig.show()

---
# 🏆 Part 3: How many wins does it take to make the playoffs?
Now we zoom out to the big goal. DePodesta looked at past seasons and estimated that **about 95 wins** almost guarantees a playoff spot.

👀 In the plot below, each dot is a team-season, coloured by whether it made the playoffs. The red line is at **95 wins**. Notice that **nearly every team to the right of the line made the playoffs**, and teams to the left are hit or miss.

In [ ]:
fig = px.scatter(df, x='wins', y='team', color=df['playoffs'].map({0: 'No', 1: 'Yes'}),
                 hover_data=['year'], labels={'color': 'Made playoffs'}, height=800)
fig.add_vline(x=95, line_color='red')
fig.show()

# HOW TO WIN 95 GAMES...

### Turn "95 wins" into a run-differential target
A GM can't buy "wins" directly, but he *can* build a roster that scores more runs than it allows. So let's model **wins as a function of run differential**:

$$\text{Wins} = b_0 + b_1 \times \text{RunDiff}$$

👀 **Look for:** R² is about 0.88, so run differential explains most of the difference in wins between teams. Notice the intercept is **≈ 81 wins**. A team that scores exactly as many runs as it allows wins about half its 162 games. That's a nice sanity check!

In [ ]:
what_we_are_trying_to_predict = 'wins'
data_we_have_related_to_games = ['runs_diff']
model = do_moneyball(df, data_we_have_related_to_games, what_we_are_trying_to_predict)

Set wins = 95 and solve the line for run differential:

$$95 = 80.88 + 0.1058 \times \text{runs\_diff} \quad\Rightarrow\quad \text{runs\_diff} = \frac{95 - 80.88}{0.1058} \approx 133$$

**The A's need to outscore their opponents by roughly 133–135 runs over the season.** DePodesta reached essentially the same number. Now the problem is concrete: build a roster that scores about 135 more runs than it allows.

---
# 🔮 Part 4: Building a playoff roster, step by step
Here's DePodesta's full method, working **backwards** from the goal. Each step is a model from this lecture or this week's exercise:

```
                 Make the playoffs
                         ↑   ~95 wins almost guarantees a spot (wins chart)
                   Win 95 games
                         ↑   wins model: 95 = 80.88 + 0.1058 × runs_diff
          Outscore opponents by ~133 runs
                         ↑   runs_diff = runs scored − runs allowed
              ┌──────────┴──────────┐
       Score more runs        Allow fewer runs
              ↑                     ↑
     runs-scored model:     runs-allowed model:
     team OBP and SLG       opponents' OBP and SLG
                            (this week's exercise)
              ↑                     ↑
     Hitters who get on     Pitchers and fielders who
     base, bought cheaply   keep opponents off base
     (quadrant chart)
```

### Predicting the 2002 A's
In the 2001–02 off-season, DePodesta could estimate the 2002 roster's stats from its players' **2001** seasons: team **OBP .339** and **SLG .430**, opponents' **OBP .307** and **SLG .373**. Plugging those into the models:

1. **Runs-scored model** → about **805 runs scored**
2. **Runs-allowed model** → about **622 runs allowed**
3. **Run differential** → 805 − 622 ≈ **+183**, well above the 133-run target
4. **Wins model** → 80.88 + 0.1058 × 183 ≈ **100 wins**

### 🎉 How did the model do?

| | **Our model's prediction** | **What actually happened in 2002** |
|---|---|---|
| Runs scored | ~805 | 800 |
| Runs allowed | ~622 | 653 |
| Wins | ~100 | **103** |

Using a few simple linear regressions and data from **before the season started**, we predicted the A's would win about 100 games. They won **103**, made the playoffs, and set an American League record with a **20-game winning streak**, all on one of the smallest payrolls in baseball.

> 🧠 **Takeaway for data scientists:** the models weren't fancy. The win came from asking the **right question**, choosing **sensible variables**, and **trusting the evidence** over conventional wisdom.

> ⚾ **Epilogue:** the A's made the playoffs in 2002 but lost in the first round. Beane famously argued that the playoffs are largely luck: a short series doesn't give skill enough games to show itself. Statistically, that's the **sample size** problem: 162 games tell you a lot; 5 games tell you very little.